In [ ]:
from stock_price_ingestor import StockPriceIngestor
from config_loader import ConfigLoader
from custom_logger import CustomLogger
from tickers import Tickers
import pandas as pd
import datetime as dt
import requests
from bs4 import BeautifulSoup


In [ ]:
def get_nyse_tickers(exchange='N'):
    url = "ftp://ftp.nasdaqtrader.com/SymbolDirectory/otherlisted.txt"
    df = pd.read_csv(url, sep="|")
    nyse_df = df[df["Exchange"] == exchange]
    return sorted(nyse_df["ACT Symbol"].dropna().unique())

def get_dow30_tickers():
    return [
        "AAPL",  # Apple
        "AMGN",  # Amgen
        "AXP",   # American Express
        "BA",    # Boeing
        "CAT",   # Caterpillar
        "CRM",   # Salesforce
        "CSCO",  # Cisco Systems
        "CVX",   # Chevron
        "DIS",   # Walt Disney
        "DOW",   # Dow Inc.
        "GS",    # Goldman Sachs
        "HD",    # Home Depot
        "HON",   # Honeywell
        "IBM",   # IBM
        "INTC",  # Intel
        "JNJ",   # Johnson & Johnson
        "JPM",   # JPMorgan Chase
        "KO",    # Coca-Cola
        "MCD",   # McDonald's
        "MMM",   # 3M
        "MRK",   # Merck
        "MSFT",  # Microsoft
        "NKE",   # Nike
        "PG",    # Procter & Gamble
        "TRV",   # Travelers
        "UNH",   # UnitedHealth
        "V",     # Visa
        "VZ",    # Verizon
        "WBA",   # Walgreens Boots Alliance
        "WMT",   # Walmart
    ]

def get_sp500_tickers():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    tables = pd.read_html(url)
    df = tables[0]
    return df["Symbol"].tolist()

def get_russell2000_tickers():
    url = "https://www.ishares.com/us/products/239710/ishares-russell-2000-etf/1467271812596.ajax?fileType=csv"
    df = pd.read_csv(url, skiprows=10)
    return sorted(df["Ticker"].dropna().unique())

def get_nasdaq_tickers():
    url = "ftp://ftp.nasdaqtrader.com/SymbolDirectory/nasdaqlisted.txt"
    df = pd.read_csv(url, sep="|")
    return df["Symbol"].tolist()

def get_nasdaq100_tickers():
    url = "https://api.nasdaq.com/api/quote/list-type/nasdaq100"
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
        "Accept-Language": "en-US,en;q=0.9"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    data = response.json()
    rows = data["data"]["data"]["rows"]
    tickers = [row["symbol"] for row in rows]
    return tickers


In [ ]:
%%time
print('Starting downloading tickers...')

nyse_tickers = get_nyse_tickers('N')
print(f'{len(nyse_tickers)=} {nyse_tickers=}')
nyse_american_tickers = get_nyse_tickers('A')
print(f'{len(nyse_american_tickers)=} {nyse_american_tickers=}')
nyse_arca_tickers = get_nyse_tickers('P')
print(f'{len(nyse_arca_tickers)=} {nyse_arca_tickers=}')

dow30_tickers = get_dow30_tickers()
print(f'{len(dow30_tickers)=} {dow30_tickers=}')

sp500_tickers = get_sp500_tickers()
print(f'{len(sp500_tickers)=} {sp500_tickers=}')

# russell2000_tickers = get_russell2000_tickers()
# print(f'{len(russell2000_tickers)=} {russell2000_tickers=}')

nasdaq_tickers = get_nasdaq_tickers()
print(f'{len(nasdaq_tickers)=} {nasdaq_tickers=}')

nasdaq100_tickers = get_nasdaq100_tickers()
print(f'{len(nasdaq100_tickers)=} {nasdaq100_tickers=}')

In [ ]:
config = ConfigLoader().get()
config

In [ ]:
custom_logger = CustomLogger(
    name='stock_price_logger', 
    log_to_console=config['log']['console'],
    log_level=config['log']['level'], 
    log_dir=config['log']['dir'], 
    log_filename=config['log']['filename'], 
)
logger = custom_logger.get_logger()
logger.info("Starting stock price ingestion process...")

In [ ]:
tickers = Tickers(logger)
print(len(tickers.get_all_tickers()))
print(f'{len(tickers.nsye_tickers)=} {tickers.nsye_tickers=}')
print(f'{len(tickers.nyse_american_tickers)=} {tickers.nyse_american_tickers=}')
print(f'{len(tickers.nyse_arca_tickers)=} {tickers.nyse_arca_tickers=}')
print(f'{len(tickers.sp500_tickers)=} {tickers.sp500_tickers=}')
print(f'{len(tickers.dow30_tickers)=} {tickers.dow30_tickers=}')
print(f'{len(tickers.russell2000_tickers)=} {tickers.russell2000_tickers=}')
print(f'{len(tickers.nasdaq_tickers)=} {tickers.nasdaq_tickers=}')
print(f'{len(tickers.nasdaq100_tickers)=} {tickers.nasdaq100_tickers=}')
print(f'{len(tickers.all_tickers)=} {tickers.all_tickers=}')

In [ ]:
db_config = config['db']
start_date = config['start_date']
ingestor = StockPriceIngestor(
    db_config=db_config,
    start_date=start_date,
    logger=logger,
)

In [ ]:
%%time 
all_tickers = tickers.all_tickers
normalized_tickers = [t.replace(".", "-") for t in all_tickers]
normalized_tickers = [t.replace("$", "-") for t in normalized_tickers]

ingestor.download(tickers=normalized_tickers)